# Prepare QE Input From The Latest BCC Dataset Frame

This notebook does the three requested steps in one place:

1. Load the newest BCC `.npz` in `dataset/bcc/non-mag` and extract the last frame.
2. Generate a zero-net noncollinear paramagnetic spin pattern in QE style. By default the notebook uses quasi-random spherical directions; paired-random is kept as an optional fallback mode.
3. Generate Maxwell-Boltzmann velocities and patch an `ATOMIC_VELOCITIES { a.u }` card into the QE input.

By default it reads the active external BCC dataset in `../dataset/bcc/non-mag` and writes the generated QE files into `../prepared_qe_inputs/latest_bcc`. Override the parameters below if you want a different dataset point or output location.

In [ ]:
from pathlib import Path
from pprint import pprint
import importlib
import sys

CWD = Path.cwd().resolve()
REPO_ROOT = None
for path in (CWD, *CWD.parents):
    direct_candidate = path
    nested_candidate = path / "IronCoreMD"
    if (direct_candidate / "codes" / "prepare_latest_bcc_qe_input.py").exists():
        REPO_ROOT = direct_candidate
        break
    if (nested_candidate / "codes" / "prepare_latest_bcc_qe_input.py").exists():
        REPO_ROOT = nested_candidate
        break
if REPO_ROOT is None:
    raise FileNotFoundError("Could not locate the IronCoreMD repo root from the current working directory.")
WORKSPACE_ROOT = REPO_ROOT.parent
CODES_DIR = REPO_ROOT / "codes"
if str(CODES_DIR) in sys.path:
    sys.path.remove(str(CODES_DIR))
sys.path.insert(0, str(CODES_DIR))

import prepare_latest_bcc_qe_input as latest_bcc_qe_input
latest_bcc_qe_input = importlib.reload(latest_bcc_qe_input)
infer_latest_bcc_npz = latest_bcc_qe_input.infer_latest_bcc_npz
prepare_bcc_qe_input = latest_bcc_qe_input.prepare_bcc_qe_input
SPIN_MODE_RANDOM_ZERO_NET = latest_bcc_qe_input.SPIN_MODE_RANDOM_ZERO_NET
SPIN_MODE_QUASI_RANDOM_ZERO_NET = latest_bcc_qe_input.SPIN_MODE_QUASI_RANDOM_ZERO_NET

DATASET_DIR = WORKSPACE_ROOT / "dataset" / "bcc" / "non-mag"
NPZ_PATH = None  # Example override: DATASET_DIR / "2.55_4000K.npz"
OUTPUT_DIR = WORKSPACE_ROOT / "prepared_qe_inputs" / "latest_bcc"

FRAME_INDEX = -1          # -1 means the last frame in the trajectory
TEMPERATURE_K = None      # None -> infer from the file name, e.g. 4000K
VELOCITY_SEED = 4000
SPIN_SEED = 123
SPIN_MODE = SPIN_MODE_QUASI_RANDOM_ZERO_NET  # default: quasi-random; optional fallback: SPIN_MODE_RANDOM_ZERO_NET
STARTING_MAGNETIZATION = 0.35

PSEUDO_DIR = WORKSPACE_ROOT
PSEUDO_FILE = "Fe.pbe-spn-kjpaw_psl.1.0.0.UPF"

DT_AU = 20.670
NSTEP = 400
ECUTWFC = 71.0
ECUTRHO = 496.0
DEGAUSS = 0.02
K_GRID = (4, 4, 4)

CONSTRAINED_MAGNETIZATION = True
LAMBDA_VALUE = 0.2
MIXING_BETA = 0.01
REMOVE_COM_DRIFT = True
RESCALE_EXACT = True
NOSYM = True

chosen_npz = NPZ_PATH if NPZ_PATH is not None else infer_latest_bcc_npz(DATASET_DIR)
print(f"Repo root:   {REPO_ROOT}")
print(f"Workspace:   {WORKSPACE_ROOT}")
print(f"Chosen NPZ:  {chosen_npz}")
print(f"Pseudo dir:  {PSEUDO_DIR}")
print(f"Output dir:  {OUTPUT_DIR.resolve()}")
print(f"Spin mode:   {SPIN_MODE}")

In [ ]:
result = prepare_bcc_qe_input(
    dataset_dir=DATASET_DIR,
    npz_path=chosen_npz,
    output_dir=OUTPUT_DIR,
    frame_index=FRAME_INDEX,
    temperature_k=TEMPERATURE_K,
    velocity_seed=VELOCITY_SEED,
    spin_seed=SPIN_SEED,
    spin_mode=SPIN_MODE,
    m_abs=STARTING_MAGNETIZATION,
    pseudo_dir=str(PSEUDO_DIR),
    pseudo_file=PSEUDO_FILE,
    dt_au=DT_AU,
    nstep=NSTEP,
    ecutwfc=ECUTWFC,
    ecutrho=ECUTRHO,
    degauss=DEGAUSS,
    k_grid=K_GRID,
    constrained_magnetization=CONSTRAINED_MAGNETIZATION,
    lambda_value=LAMBDA_VALUE,
    mixing_beta=MIXING_BETA,
    remove_com_drift=REMOVE_COM_DRIFT,
    rescale_exact=RESCALE_EXACT,
    nosym=NOSYM,
)

pprint(result.as_dict())

In [ ]:
final_input_path = Path(result.qe_input_final)
velocity_block_path = Path(result.velocity_block)
spin_parameters_path = Path(result.spin_parameters)

print("Final QE input preview:\n")
print("\n".join(final_input_path.read_text().splitlines()[:60]))

print("\nVelocity card preview:\n")
print("\n".join(velocity_block_path.read_text().splitlines()[:8]))

print("\nSpin parameter preview:\n")
print("\n".join(spin_parameters_path.read_text().splitlines()[:12]))

In [ ]:
def extract_card_labels_from_file(path: Path, card_name: str, min_fields: int) -> list[str]:
    lines = path.read_text().splitlines()
    card_upper = card_name.upper()
    card_prefixes = (
        "&CONTROL",
        "&SYSTEM",
        "&ELECTRONS",
        "&IONS",
        "&CELL",
        "ATOMIC_SPECIES",
        "ATOMIC_POSITIONS",
        "ATOMIC_VELOCITIES",
        "K_POINTS",
        "CELL_PARAMETERS",
        "CONSTRAINTS",
        "OCCUPATIONS",
        "ATOMIC_FORCES",
        "SOLVENTS",
        "HUBBARD",
    )

    start = None
    for index, line in enumerate(lines):
        if line.strip().upper().startswith(card_upper):
            start = index
            break
    if start is None:
        raise ValueError(f"Could not find {card_name} in {path}")

    end = start + 1
    while end < len(lines):
        stripped = lines[end].strip().upper()
        if stripped.startswith(card_prefixes):
            break
        end += 1

    labels = []
    for line in lines[start + 1 : end]:
        stripped = line.strip()
        if not stripped or stripped.startswith(("!", "#")):
            continue
        fields = stripped.split()
        if len(fields) < min_fields:
            continue
        labels.append(fields[0])
    return labels


generated_input = Path(result.qe_input_final)
species_labels = extract_card_labels_from_file(generated_input, "ATOMIC_SPECIES", 3)
position_labels = extract_card_labels_from_file(generated_input, "ATOMIC_POSITIONS", 4)
velocity_labels = extract_card_labels_from_file(generated_input, "ATOMIC_VELOCITIES", 4)

line_by_line_match = position_labels == velocity_labels
if not line_by_line_match:
    mismatches = [
        (index + 1, pos_label, vel_label)
        for index, (pos_label, vel_label) in enumerate(zip(position_labels, velocity_labels))
        if pos_label != vel_label
    ]
    raise ValueError(f"Position/velocity label mismatch in generated QE input: {mismatches[:5]}")

species_set = set(species_labels)
if any(label not in species_set for label in position_labels):
    raise ValueError("Some ATOMIC_POSITIONS labels are missing from ATOMIC_SPECIES")
if any(label not in species_set for label in velocity_labels):
    raise ValueError("Some ATOMIC_VELOCITIES labels are missing from ATOMIC_SPECIES")

print(f"Generated input file: {generated_input}")
print(f"ATOMIC_SPECIES count: {len(species_labels)}")
print(f"ATOMIC_POSITIONS count: {len(position_labels)}")
print(f"ATOMIC_VELOCITIES count: {len(velocity_labels)}")
print(f"ATOMIC_POSITIONS and ATOMIC_VELOCITIES match line-by-line: {line_by_line_match}")
print(f"First 10 labels: {position_labels[:10]}")
print(f"Last 10 labels: {position_labels[-10:]}")
